# Phase 1 : Encodage des Images du Dataset

Ce notebook génère les embeddings pour tout le dataset de vêtements en utilisant des modèles pré-entraînés.

In [ ]:
import os
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms

# Configuration
DATASET_ROOT = Path('../clothing-dataset-small-master')
EMBEDDINGS_DIR = Path('../embeddings')
EMBEDDINGS_DIR.mkdir(exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Image transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Using device: cpu


## 1. Explorer le Dataset

In [ ]:
# Parcourir et lister les images du dataset
image_paths = []
image_labels = []

train_dir = DATASET_ROOT / 'train'

for category_dir in sorted(train_dir.iterdir()):
    if category_dir.is_dir():
        category = category_dir.name
        category_images = list(category_dir.glob('*.jpg')) + list(category_dir.glob('*.jpeg')) + list(category_dir.glob('*.png'))
        print(f'{category}: {len(category_images)} images')
        
        for img_path in category_images:
            image_paths.append(str(img_path))
            image_labels.append(category)

print(f'\nTotal: {len(image_paths)} images')

dress: 241 images
hat: 123 images
longsleeve: 455 images
outwear: 184 images
pants: 468 images
shirt: 290 images
shoes: 198 images
shorts: 202 images
skirt: 112 images
t-shirt: 795 images

Total: 3068 images


## 2. Définir la Classe d'Encodeur

In [ ]:
class ImageEncoder:
    def __init__(self, model_name='resnet18', device='cpu'):
        self.device = device
        self.model_name = model_name
        self.model = self._load_model(model_name)
        self.embedding_dim = self._get_embedding_dim()
        
    def _load_model(self, model_name):
        if model_name == 'resnet18':
            model = models.resnet18(pretrained=True)
        elif model_name == 'mobilenetv2':
            model = models.mobilenet_v2(pretrained=True)
        else:
            raise ValueError(f'Unknown model: {model_name}')
        
        # Remove classification layer, keep features
        model = torch.nn.Sequential(*list(model.children())[:-1])
        model.to(self.device)
        model.eval()
        return model
    
    def _get_embedding_dim(self):
        # Create dummy input to infer embedding dimension
        dummy_input = torch.randn(1, 3, 224, 224).to(self.device)
        with torch.no_grad():
            output = self.model(dummy_input)
        return output.shape[1]
    
    def encode_image(self, image_path):
        try:
            img = Image.open(image_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                embedding = self.model(img_tensor)
            
            # Flatten and normalize
            embedding = embedding.view(embedding.size(0), -1)
            embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)
            return embedding.cpu().numpy().flatten()
        except Exception as e:
            print(f'Error encoding {image_path}: {e}')
            return None
    
    def encode_batch(self, image_paths, batch_size=32):
        embeddings = []
        
        for i in tqdm(range(0, len(image_paths), batch_size), desc=f'Encoding with {self.model_name}'):
            batch_paths = image_paths[i:i+batch_size]
            batch_embeddings = []
            
            for path in batch_paths:
                emb = self.encode_image(path)
                if emb is not None:
                    batch_embeddings.append(emb)
            
            embeddings.extend(batch_embeddings)
        
        return np.array(embeddings)

print('Classe ImageEncoder définie')

## 3. Générer les Embeddings avec ResNet-18

In [ ]:
# Encoder with ResNet-18
encoder_resnet = ImageEncoder('resnet18', device=device)
print(f'ResNet-18 embedding dimension: {encoder_resnet.embedding_dim}')

embeddings_resnet = encoder_resnet.encode_batch(image_paths, batch_size=32)
print(f'Generated {embeddings_resnet.shape[0]} embeddings of dimension {embeddings_resnet.shape[1]}')

# Save embeddings
np.save(EMBEDDINGS_DIR / 'resnet18_embeddings.npy', embeddings_resnet)
print('Saved: resnet18_embeddings.npy')

## 4. Générer les Embeddings avec MobileNetV2

In [ ]:
# Encoder with MobileNetV2
encoder_mobile = ImageEncoder('mobilenetv2', device=device)
print(f'MobileNetV2 embedding dimension: {encoder_mobile.embedding_dim}')

embeddings_mobile = encoder_mobile.encode_batch(image_paths, batch_size=32)
print(f'Generated {embeddings_mobile.shape[0]} embeddings of dimension {embeddings_mobile.shape[1]}')

# Save embeddings
np.save(EMBEDDINGS_DIR / 'mobilenetv2_embeddings.npy', embeddings_mobile)
print('Saved: mobilenetv2_embeddings.npy')

## 5. Sauvegarder les Métadonnées

In [ ]:
# Create metadata file
metadata = {
    'total_images': len(image_paths),
    'image_paths': image_paths,
    'image_labels': image_labels,
    'models': {
        'resnet18': {
            'embedding_dim': int(encoder_resnet.embedding_dim),
            'embeddings_file': 'resnet18_embeddings.npy',
            'num_embeddings': int(embeddings_resnet.shape[0])
        },
        'mobilenetv2': {
            'embedding_dim': int(encoder_mobile.embedding_dim),
            'embeddings_file': 'mobilenetv2_embeddings.npy',
            'num_embeddings': int(embeddings_mobile.shape[0])
        }
    }
}

with open(EMBEDDINGS_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: metadata.json')
print(f'\nSummary:')
print(f'  Total images: {metadata["total_images"]}')
print(f'  Categories: {len(set(image_labels))}')
print(f'  ResNet-18 embeddings: {embeddings_resnet.shape}')
print(f'  MobileNetV2 embeddings: {embeddings_mobile.shape}')

## Encodage Terminé

Les embeddings ont été générés et sauvegardés avec succès.
Allez au notebook `02_requete.ipynb` pour tester la recherche d'images similaires.